# Mini Competition – Analysis & Visualization
## IMLS Public Libraries Survey (PLS) FY 2022
**Files used:** `PLS_FY22_AE_pud22i.csv` (9,248 library systems – *imputed* version) and `pls_fy22_outlet_pud22i.csv` (17,546 outlets).

**Workflow:** run one cell → take a screenshot of the code + output → move to the next cell.
Cells are grouped exactly as per the four phases in the guidelines.

# PHASE 1 — Dataset Understanding & Data Preprocessing

### 1.0  Setup & load data

In [ ]:
import warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial import distance
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

DATA_DIR = "."          # <-- change if your CSVs are in another folder
ae     = pd.read_csv(f"{DATA_DIR}/PLS_FY22_AE_pud22i.csv",     encoding="latin-1", low_memory=False)
outlet = pd.read_csv(f"{DATA_DIR}/pls_fy22_outlet_pud22i.csv", encoding="latin-1", low_memory=False)

print("Library-system (AE) file :", ae.shape)
print("Outlet file              :", outlet.shape)
display(ae[["STABR","LIBNAME","CITY","POPU_LSA","TOTSTAFF","TOTOPEXP","VISITS","TOTCIR","LOCALE_ADD"]].head())

### 1.1  Attribute identification (Nominal / Ordinal / Interval / Ratio, Discrete / Continuous)

In [ ]:
attr = [
 # column,      meaning,                              scale,     kind,       allowed operations
 ("STABR",      "State abbreviation",                 "Nominal",  "Categorical", "mode, counts, group-by"),
 ("LIBNAME",    "Library name",                       "Nominal",  "Text",        "edit / cosine distance"),
 ("CNTY",       "County",                             "Nominal",  "Categorical", "mode, counts"),
 ("C_ADMIN",    "Administrative structure code",      "Nominal",  "Categorical", "mode, counts"),
 ("C_OUT_TY",   "Outlet type (CE/BR/BS/BM) [outlet]","Nominal",  "Categorical", "mode, counts"),
 ("LOCALE_ADD", "Urban-centric locale code (11-43)",  "Ordinal",  "Categorical", "median, mode, rank"),
 ("YR_SUB",     "Year of data submission",            "Interval", "Discrete",    "differences (no true zero)"),
 ("LATITUDE",   "Latitude of library",                "Interval", "Continuous",  "mean, difference, distance"),
 ("LONGITUD",   "Longitude of library",               "Interval", "Continuous",  "mean, difference, distance"),
 ("POPU_LSA",   "Legal service-area population",      "Ratio",    "Discrete",    "all (ratios meaningful)"),
 ("CENTLIB",    "No. of central libraries",           "Ratio",    "Discrete",    "all"),
 ("BRANLIB",    "No. of branch libraries",            "Ratio",    "Discrete",    "all"),
 ("TOTSTAFF",   "Total paid staff (FTE)",             "Ratio",    "Continuous",  "all"),
 ("TOTINCM",    "Total operating revenue (USD)",      "Ratio",    "Continuous",  "all"),
 ("TOTOPEXP",   "Total operating expenditure (USD)",  "Ratio",    "Continuous",  "all"),
 ("BKVOL",      "Print books held",                   "Ratio",    "Discrete",    "all"),
 ("TOTCIR",     "Total circulation (loans)",          "Ratio",    "Discrete",    "all"),
 ("VISITS",     "Annual in-person visits",            "Ratio",    "Discrete",    "all"),
 ("TOTPRO",     "Total programs held",                "Ratio",    "Discrete",    "all"),
 ("TOTATTEN",   "Total program attendance",           "Ratio",    "Discrete",    "all"),
 ("HRS_OPEN",   "Total public hours open / year",     "Ratio",    "Continuous",  "all"),
 ("WIFISESS",   "Wi-Fi sessions",                     "Ratio",    "Discrete",    "all"),
 ("PITUSR",     "Public Internet-computer uses",      "Ratio",    "Discrete",    "all"),
 ("SQ_FEET",    "Outlet floor area [outlet]",         "Ratio",    "Continuous",  "all"),
 ("STARTDAT",   "Fiscal-year start date",             "Interval", "Date",        "differences"),
]
attr_df = pd.DataFrame(attr, columns=["Attribute","Meaning","Scale","Type","Suitable operations"])
display(attr_df)
print("\nAutomatic dtype summary of the AE file:")
display(ae.dtypes.astype(str).value_counts().rename("n_columns").to_frame())

### 1.2  Data-quality assessment (missing codes, duplicates, inconsistencies)

In [ ]:
# --- PLS encodes missing data with negative codes (see README): -1 missing, -3 closed, -4 not applicable, -9 suppressed
codes = {-1: "Missing", -3: "Closed", -4: "Not applicable", -9: "Suppressed"}
num = ae.select_dtypes("number")
neg = pd.DataFrame({name: (num == c).sum() for c, name in codes.items()})
neg["Total coded"] = neg.sum(axis=1)
print("Columns holding special missing codes (top 12):")
display(neg[neg["Total coded"] > 0].sort_values("Total coded", ascending=False).head(12))

# character 'M' also means missing
chr_cols = ae.select_dtypes("object").columns
m_cnt = (ae[chr_cols] == "M").sum().sort_values(ascending=False)
print("\nText columns containing the missing marker 'M' (top 8):")
display(m_cnt[m_cnt > 0].head(8).to_frame("count_of_M"))

# --- Duplicates
print("\nFully duplicated rows (AE)        :", ae.duplicated().sum())
print("Duplicated LIBID (AE)             :", ae.LIBID.duplicated().sum())
print("Duplicated FSCSKEY+FSCS_SEQ (outlet):", outlet.duplicated(["FSCSKEY","FSCS_SEQ"]).sum())

# --- Entry inconsistencies
inc = {
 "Library name has leading/trailing spaces"            : (ae.LIBNAME != ae.LIBNAME.str.strip()).sum(),
 "State code not 2 letters"                            : (ae.STABR.str.len() != 2).sum(),
 "Service-area population <= 0"                        : (ae.POPU_LSA <= 0).sum(),
 "Temporarily closed (STATSTRU = 23)"                  : (ae.STATSTRU == 23).sum(),
 "TOTINCM != sum of its 4 revenue parts"               : (ae.TOTINCM != ae[["LOCGVT","STGVT","FEDGVT","OTHINCM"]].sum(axis=1)).sum(),
 "TOTCIR < physical circulation"                       : (ae.TOTCIR < ae.PHYSCIR).sum(),
 "VISITS > 0 but HRS_OPEN = 0"                         : ((ae.VISITS > 0) & (ae.HRS_OPEN == 0)).sum(),
 "TOTATTEN > 0 but TOTPRO = 0"                         : ((ae.TOTATTEN > 0) & (ae.TOTPRO == 0)).sum(),
}
print("\nInconsistency checks")
display(pd.Series(inc, name="rows affected").to_frame())

# --- How much of the *imputed* file was actually imputed?  (flags start with 'I' when imputed)
flag_cols = [c for c in ae.columns if c.startswith("F_")]
imp_share = pd.Series({c: ae[c].astype(str).str.startswith("I").mean()*100 for c in flag_cols}).sort_values(ascending=False)
print("\n% of values that the IMLS imputed, by variable flag (top 8):")
display(imp_share.head(8).round(2).to_frame("% imputed"))

### 1.3  Clean the data & visualise remaining missingness

In [ ]:
# Work only on libraries that were open
df = ae[ae.STATSTRU != 23].copy()

NUM = ["POPU_LSA","CENTLIB","BRANLIB","BKMOB","TOTSTAFF","TOTINCM","SALARIES","BENEFIT","STAFFEXP","OTHOPEXP",
       "TOTOPEXP","BKVOL","TOTCIR","VISITS","REGBOR","TOTPRO","TOTATTEN","HRS_OPEN","WIFISESS","PITUSR",
       "ONPRO","OFFPRO","VIRPRO","EBOOK","AUDIO_DL","VIDEO_DL","LATITUDE","LONGITUD"]

df[NUM] = df[NUM].replace([-1, -3, -4, -9], np.nan)                # special codes -> NaN
df["LOCALE_ADD"] = df["LOCALE_ADD"].replace([-1,-3,-4,-9], np.nan)

miss = (df[NUM].isna().mean()*100).sort_values(ascending=False)
display(miss[miss > 0].round(2).to_frame("% missing"))

fig, ax = plt.subplots(figsize=(9, 4))
miss[miss > 0].plot.bar(ax=ax, color="tomato")
ax.set_ylabel("% missing"); ax.set_title("Missing values after converting PLS codes to NaN")
plt.tight_layout(); plt.show()
print("Rows kept:", len(df))

### 1.4  Missing-data handling – compare Median vs Regression vs Interpolation

In [ ]:
# Variable with real gaps: STAFFEXP (staff expenditure) is suppressed (-9) for tiny libraries.
# We test 3 techniques on the KNOWN values by hiding 20 % of them and measuring RMSE.
rng = np.random.default_rng(42)
known = df.dropna(subset=["STAFFEXP","TOTOPEXP","TOTSTAFF"]).copy()
mask = rng.random(len(known)) < 0.20
truth = known.loc[mask, "STAFFEXP"]
train = known.loc[~mask]

# (a) median imputation
pred_med = pd.Series(train.STAFFEXP.median(), index=truth.index)

# (b) regression imputation  STAFFEXP ~ TOTOPEXP + TOTSTAFF  (least squares)
X = np.c_[np.ones(len(train)), train.TOTOPEXP, train.TOTSTAFF]
beta, *_ = np.linalg.lstsq(X, train.STAFFEXP, rcond=None)
Xm = np.c_[np.ones(mask.sum()), known.loc[mask, "TOTOPEXP"], known.loc[mask, "TOTSTAFF"]]
pred_reg = pd.Series(Xm @ beta, index=truth.index)

# (c) interpolation along TOTOPEXP (sorted order)
tmp = known.sort_values("TOTOPEXP").copy()
tmp.loc[tmp.index.isin(truth.index), "STAFFEXP"] = np.nan
tmp["STAFFEXP"] = tmp["STAFFEXP"].interpolate(method="linear", limit_direction="both")
pred_int = tmp.loc[truth.index, "STAFFEXP"]

rmse = lambda p: np.sqrt(((truth - p)**2).mean())
res = pd.DataFrame({"Method": ["Median", "Regression", "Linear interpolation"],
                    "RMSE (USD)": [rmse(pred_med), rmse(pred_reg), rmse(pred_int)]}).sort_values("RMSE (USD)")
display(res)
print("Regression coefficients [intercept, TOTOPEXP, TOTSTAFF]:", beta.round(3))

fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(data=res, x="Method", y="RMSE (USD)", ax=ax, palette="viridis")
ax.set_title("Imputation error (lower is better)"); plt.tight_layout(); plt.show()

In [ ]:
# ---- Apply the chosen strategy to the full dataset
# 1. STAFFEXP: regression imputation (best RMSE above, uses all known rows)
k = df.dropna(subset=["STAFFEXP","TOTOPEXP","TOTSTAFF"])
Xk = np.c_[np.ones(len(k)), k.TOTOPEXP, k.TOTSTAFF]
b, *_ = np.linalg.lstsq(Xk, k.STAFFEXP, rcond=None)
need = df.STAFFEXP.isna() & df.TOTOPEXP.notna() & df.TOTSTAFF.notna()
df.loc[need, "STAFFEXP"] = (np.c_[np.ones(need.sum()), df.loc[need,"TOTOPEXP"], df.loc[need,"TOTSTAFF"]] @ b).clip(min=0)

# 2. everything else: median imputation within locale group (robust to skew)
df["LOCALE_GRP"] = df["LOCALE_ADD"].fillna(42).astype(int).astype(str).str[0].map({"1":"City","2":"Suburb","3":"Town","4":"Rural"})
for c in NUM:
    df[c] = df[c].fillna(df.groupby("LOCALE_GRP")[c].transform("median")).fillna(df[c].median())

print("Remaining NaN in analysis columns:", int(df[NUM].isna().sum().sum()))

# derived per-capita measures used later
df = df[df.POPU_LSA > 0].copy()
df["VISITS_PC"] = df.VISITS / df.POPU_LSA
df["CIRC_PC"]   = df.TOTCIR / df.POPU_LSA
df["SPEND_PC"]  = df.TOTOPEXP / df.POPU_LSA
df["PROG_ATT_PC"] = df.TOTATTEN / df.POPU_LSA
display(df[["VISITS_PC","CIRC_PC","SPEND_PC","PROG_ATT_PC"]].describe().T)

### 1.5  Transformation & normalisation (Min-Max, Z-score, Decimal scaling)

In [ ]:
FEATS = ["POPU_LSA","TOTSTAFF","TOTOPEXP","BKVOL","TOTCIR","VISITS","TOTPRO","TOTATTEN","HRS_OPEN","WIFISESS","PITUSR","REGBOR"]

minmax  = (df[FEATS] - df[FEATS].min()) / (df[FEATS].max() - df[FEATS].min())
zscore  = (df[FEATS] - df[FEATS].mean()) / df[FEATS].std()
j       = np.ceil(np.log10(df[FEATS].abs().max()))                 # decimal scaling: x / 10^j
decimal = df[FEATS] / (10 ** j)
logt    = np.log10(df[FEATS] + 1)                                  # extra: log transform for heavy skew

print("Decimal-scaling exponent j per feature:"); display(j.astype(int).to_frame("j").T)
summary = pd.DataFrame({
    "MinMax  min": minmax.min(), "MinMax  max": minmax.max(),
    "Z-score mean": zscore.mean(), "Z-score std": zscore.std(),
    "Decimal max": decimal.max()})
display(summary)

fig, axs = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (title, d) in zip(axs, [("Original", df["VISITS"]), ("Min-Max", minmax["VISITS"]),
                               ("Z-score", zscore["VISITS"]), ("log10(x+1)", logt["VISITS"])]):
    ax.hist(d, bins=50, color="steelblue"); ax.set_title(f"VISITS – {title}")
plt.tight_layout(); plt.show()

### 1.6  Discretisation & binning (equal-width, equal-frequency, histogram analysis)

In [ ]:
pop = df["POPU_LSA"]

# equal-width : 5 bins of identical range
ew = pd.cut(pop, bins=5)
# equal-frequency : quintiles with ~same number of libraries
ef = pd.qcut(pop, q=5, duplicates="drop")
# histogram analysis : data-driven bin edges (Freedman–Diaconis rule) on log10 scale
edges = np.histogram_bin_edges(np.log10(pop), bins="fd")
ha = pd.cut(np.log10(pop), bins=edges)

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for ax, (t, s) in zip(axs, [("Equal-width (5 bins)", ew), ("Equal-frequency (5 bins)", ef), ("Histogram / FD rule on log10", ha)]):
    vc = s.value_counts().sort_index()
    ax.bar(range(len(vc)), vc.values, color="teal")
    ax.set_xticks(range(len(vc))); ax.set_xticklabels([str(i) for i in vc.index], rotation=70, ha="right", fontsize=7)
    ax.set_title(t); ax.set_ylabel("No. of libraries")
plt.tight_layout(); plt.show()

print("Equal-width counts     :", ew.value_counts().sort_index().values)
print("Equal-frequency counts :", ef.value_counts().sort_index().values)

# Domain-friendly categorical feature (used later in the dashboard)
df["SIZE_CLASS"] = pd.cut(pop, [0, 2_500, 10_000, 50_000, 250_000, np.inf],
                          labels=["Tiny <2.5k", "Small 2.5-10k", "Medium 10-50k", "Large 50-250k", "Metro >250k"])
display(df["SIZE_CLASS"].value_counts().sort_index().to_frame("libraries"))

# PHASE 2 — Exploratory & Statistical Data Analysis

### 2.1  Central tendency – ungrouped data (mean, weighted mean, geometric mean, median, mode)

In [ ]:
rows = {}
for c in FEATS:
    x = df[c]
    rows[c] = dict(Mean=x.mean(), Median=x.median(), Mode=x.mode().iloc[0],
                   GeoMean=stats.gmean(x[x > 0]))
ct = pd.DataFrame(rows).T
display(ct)

# Weighted mean: national visits-per-person. Simple mean treats a village and a metropolis equally;
# weighting by population gives the true per-resident figure.
simple_mean   = df.VISITS_PC.mean()
weighted_mean = np.average(df.VISITS_PC, weights=df.POPU_LSA)
print(f"Visits per capita – simple mean   : {simple_mean:.3f}")
print(f"Visits per capita – weighted mean : {weighted_mean:.3f}   (weights = service-area population)")

for c in ["LOCALE_GRP", "SIZE_CLASS"]:
    print(f"\nMode of {c}:", df[c].mode().iloc[0])

### 2.2  Central tendency – grouped data (class mid-points)

In [ ]:
x = df["HRS_OPEN"]
x = x[x <= x.quantile(0.99)]                                 # trim extreme top 1 % for readable classes
h = 500                                                       # class width
bins = np.arange(0, x.max() + h, h)
f, edges = np.histogram(x, bins=bins)
g = pd.DataFrame({"Class": [f"{int(a)}-{int(b)}" for a, b in zip(edges[:-1], edges[1:])],
                  "L": edges[:-1], "f": f})
g["Midpoint m"] = g.L + h/2
g["f*m"] = g.f * g["Midpoint m"]
g["CF"]  = g.f.cumsum()
display(g)

N = g.f.sum()
grp_mean = g["f*m"].sum() / N
mc = g.index[g.CF >= N/2][0]                                  # median class
grp_median = g.L[mc] + ((N/2 - (g.CF[mc] - g.f[mc])) / g.f[mc]) * h
k  = g.f.idxmax()                                             # modal class
d1 = g.f[k] - (g.f[k-1] if k > 0 else 0); d2 = g.f[k] - (g.f[k+1] if k < len(g)-1 else 0)
grp_mode = g.L[k] + d1/(d1 + d2) * h

cmp_ = pd.DataFrame({"Grouped formula": [grp_mean, grp_median, grp_mode],
                     "Ungrouped (raw)": [x.mean(), x.median(), x.round(-1).mode().iloc[0]]},
                    index=["Mean", "Median", "Mode"])
display(cmp_)

### 2.3  Measures of dispersion

In [ ]:
disp = {}
for c in FEATS:
    x = df[c]; q1, q3 = x.quantile([.25, .75])
    disp[c] = dict(Range=x.max()-x.min(), Variance=x.var(), Std=x.std(),
                   RMSD=np.sqrt(((x - x.mean())**2).mean()),       # root-mean-square deviation about the mean
                   CV=x.std()/x.mean(),                              # coefficient of variation
                   QuartileCoefDisp=(q3-q1)/(q3+q1))                 # coefficient of dispersion (robust)
dp = pd.DataFrame(disp).T
display(dp.style.format("{:,.3f}"))

### 2.4  Shape measures (Pearson & Bowley skewness, kurtosis) and IQR outliers

In [ ]:
shape = {}
for c in FEATS:
    x = df[c]; q1, q2, q3 = x.quantile([.25, .5, .75])
    iqr = q3 - q1; lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    out = ((x < lo) | (x > hi)).sum()
    shape[c] = dict(Pearson2=3*(x.mean()-q2)/x.std(), Bowley=(q3+q1-2*q2)/(q3-q1),
                    MomentSkew=stats.skew(x), ExcessKurtosis=stats.kurtosis(x),
                    Q1=q1, Q3=q3, IQR=iqr, LowerFence=lo, UpperFence=hi,
                    Outliers=out, OutlierPct=out/len(x)*100)
sh = pd.DataFrame(shape).T
display(sh.style.format("{:,.2f}"))

# effect of log transform on skewness
comp = pd.DataFrame({"Skew raw": [stats.skew(df[c]) for c in FEATS],
                     "Skew log10": [stats.skew(np.log10(df[c]+1)) for c in FEATS]}, index=FEATS)
display(comp.style.format("{:,.2f}"))

### 2.5  Distance & proximity – numeric (Euclidean, Manhattan, Minkowski, Mahalanobis)

In [ ]:
DF5 = ["POPU_LSA","TOTSTAFF","TOTOPEXP","TOTCIR","VISITS","HRS_OPEN"]
Z = (np.log10(df[DF5] + 1) - np.log10(df[DF5] + 1).mean()) / np.log10(df[DF5] + 1).std()

# five libraries of increasing size (10th, 30th, 50th, 70th, 90th percentile of population)
order = df.POPU_LSA.sort_values()
idx = [order.index[int(len(order)*p)] for p in (.10, .30, .50, .70, .90)]
labels = [df.loc[i, "LIBNAME"].title()[:22] for i in idx]
S = Z.loc[idx].values
print("Libraries compared:", labels)

VI = np.linalg.inv(np.cov(Z.values.T))
mats = {
 "Euclidean"        : distance.cdist(S, S, "euclidean"),
 "Manhattan"        : distance.cdist(S, S, "cityblock"),
 "Minkowski (p=3)"  : distance.cdist(S, S, "minkowski", p=3),
 "Mahalanobis"      : distance.cdist(S, S, "mahalanobis", VI=VI),
}
for name, m in mats.items():
    print("\n", name); display(pd.DataFrame(m, index=labels, columns=labels).round(2))

### 2.6  Distance & proximity – binary (SMC / Jaccard) and textual (Cosine / Edit distance)

In [ ]:
# ---- Binary service-feature vectors (1 = library offers it)
bin_cols = {"Branches":"BRANLIB","Bookmobile":"BKMOB","E-books":"EBOOK","Audio downloads":"AUDIO_DL",
            "Video downloads":"VIDEO_DL","Online programs":"ONPRO","Off-site programs":"OFFPRO",
            "Virtual programs":"VIRPRO","Wi-Fi":"WIFISESS","Public PCs":"PITUSR"}
B = (df.loc[idx, list(bin_cols.values())] > 0).astype(int)
B.columns = list(bin_cols.keys()); B.index = labels
display(B)

def smc(a, b):  return (a == b).mean()
def jac(a, b):  return ((a & b).sum() / (a | b).sum()) if (a | b).sum() else 1.0
A = B.values.astype(bool); n = len(A)
SMC = pd.DataFrame([[smc(A[i], A[j]) for j in range(n)] for i in range(n)], index=labels, columns=labels)
JAC = pd.DataFrame([[jac(A[i], A[j]) for j in range(n)] for i in range(n)], index=labels, columns=labels)
print("Simple Matching Coefficient (similarity)"); display(SMC.round(2))
print("Jaccard coefficient (similarity, ignores 0-0 matches)"); display(JAC.round(2))

# ---- Textual: cosine on word-count vectors and Levenshtein edit distance on library names
names = df.loc[df.LIBNAME.str.contains("SPRINGFIELD"), "LIBNAME"].drop_duplicates().head(5).tolist()
def lev(a, b):
    dp_ = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        prev, dp_[0] = dp_[0], i
        for j, cb in enumerate(b, 1):
            prev, dp_[j] = dp_[j], min(dp_[j]+1, dp_[j-1]+1, prev + (ca != cb))
    return dp_[-1]
vocab = sorted({w for s in names for w in s.split()})
V = np.array([[s.split().count(w) for w in vocab] for s in names], float)
cos = 1 - distance.cdist(V, V, "cosine")
ED  = np.array([[lev(a, b) for b in names] for a in names])
short = [n_[:24] for n_ in names]
print("Cosine similarity of names");  display(pd.DataFrame(cos, index=short, columns=short).round(2))
print("Edit (Levenshtein) distance"); display(pd.DataFrame(ED,  index=short, columns=short))

# PHASE 3 — Visualisation Selection & Graphical Representation

### 3.1  Frequency distribution – Histogram, Frequency polygon, Ogive

In [ ]:
x = df["HRS_OPEN"]; x = x[(x > 0) & (x <= x.quantile(0.99))]
counts, edges = np.histogram(x, bins=30)
mid = (edges[:-1] + edges[1:]) / 2

fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
axs[0].hist(x, bins=edges, color="steelblue", edgecolor="white")
axs[0].set_title("Histogram – annual public hours open"); axs[0].set_xlabel("Hours / year"); axs[0].set_ylabel("Libraries")

axs[1].plot(np.r_[mid[0]-(mid[1]-mid[0]), mid, mid[-1]+(mid[1]-mid[0])], np.r_[0, counts, 0], marker="o", color="darkorange")
axs[1].set_title("Frequency polygon"); axs[1].set_xlabel("Hours / year")

cum = np.cumsum(counts)
axs[2].plot(edges[1:], cum, marker=".", color="seagreen")                  # 'less-than' ogive
axs[2].axhline(len(x)/2, ls="--", c="grey"); axs[2].text(edges[1], len(x)/2 + 40, "median (50 %)", color="grey")
axs[2].set_title("Cumulative frequency curve (Ogive)"); axs[2].set_xlabel("Hours / year")
plt.tight_layout(); plt.show()

### 3.2  Boxplots & Q-Q plots (spread, outliers, normality)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=np.log10(df[["POPU_LSA","TOTOPEXP","TOTCIR","VISITS","TOTATTEN","HRS_OPEN"]] + 1),
            ax=axs[0], palette="Set2", fliersize=2)
axs[0].set_title("Boxplots of key measures (log10 scale – dots = IQR outliers)"); axs[0].set_ylabel("log10(value + 1)")
axs[0].tick_params(axis="x", rotation=25)

sns.boxplot(data=df, x="LOCALE_GRP", y="VISITS_PC", order=["City","Suburb","Town","Rural"], ax=axs[1], palette="Set3", fliersize=2)
axs[1].set_yscale("log"); axs[1].set_title("Visits per resident by locale (log scale)"); axs[1].set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 9))
for r, c in enumerate(["POPU_LSA", "TOTCIR"]):
    stats.probplot(df[c], dist="norm", plot=axs[r, 0]);                   axs[r, 0].set_title(f"Q-Q plot – {c} (raw)")
    stats.probplot(np.log10(df[c] + 1), dist="norm", plot=axs[r, 1]);     axs[r, 1].set_title(f"Q-Q plot – log10({c})")
plt.tight_layout(); plt.show()
for c in ["POPU_LSA","TOTCIR"]:
    print(f"{c}: Shapiro-Wilk p (raw, n=5000 sample) = {stats.shapiro(df[c].sample(5000, random_state=1))[1]:.2e} | "
          f"after log10 = {stats.shapiro(np.log10(df[c].sample(5000, random_state=1) + 1))[1]:.2e}")

### 3.3  Relational plots – Scatter plots, Correlation heat-map, Pair-plot

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, (xc, yc) in zip(axs, [("POPU_LSA","VISITS"), ("TOTOPEXP","TOTCIR"), ("TOTSTAFF","TOTATTEN")]):
    sns.scatterplot(data=df, x=xc, y=yc, hue="LOCALE_GRP", alpha=.35, s=14, ax=ax, legend=(ax is axs[0]))
    ax.set_xscale("log"); ax.set_yscale("log")
    r = np.corrcoef(np.log10(df[xc]+1), np.log10(df[yc]+1))[0, 1]
    ax.set_title(f"{yc} vs {xc}   (log-log r = {r:.2f})")
plt.tight_layout(); plt.show()

In [ ]:
corr = df[FEATS].corr(method="spearman")
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=0, vmax=1, square=True, cbar_kws={"label": "Spearman ρ"})
plt.title("Rank correlation between library measures"); plt.tight_layout(); plt.show()

In [ ]:
pp = np.log10(df[["POPU_LSA","TOTOPEXP","TOTCIR","VISITS","TOTATTEN"]] + 1).join(df["LOCALE_GRP"])
g = sns.pairplot(pp.sample(1500, random_state=7), hue="LOCALE_GRP", corner=True, plot_kws=dict(s=12, alpha=.5), height=2.1)
g.fig.suptitle("Pair-plot (log10 scale, 1,500-library sample)", y=1.02); plt.show()

### 3.4  Outlet file – floor area & opening hours

In [ ]:
o = outlet[(outlet.STATSTRU != 23)].copy()
o[["SQ_FEET","HOURS","WKS_OPEN"]] = o[["SQ_FEET","HOURS","WKS_OPEN"]].replace([-1,-3,-4,-9], np.nan)
o = o.dropna(subset=["SQ_FEET","HOURS"]); o = o[(o.SQ_FEET > 0) & (o.HOURS > 0)]
o["TYPE"]   = o.C_OUT_TY.map({"CE":"Central","BR":"Branch","BS":"Bookmobile","BM":"Books-by-mail"})
o["LOC_GRP"]= (pd.to_numeric(o.LOCALE, errors="coerce") // 10).map({1:"City",2:"Suburb",3:"Town",4:"Rural"})
print("Outlets analysed:", len(o)); display(o.groupby("TYPE")[["SQ_FEET","HOURS"]].median().round(0))

fig, axs = plt.subplots(1, 3, figsize=(18, 4.8))
sns.boxplot(data=o[o.TYPE.isin(["Central","Branch"])], x="TYPE", y="SQ_FEET", ax=axs[0], palette="pastel"); axs[0].set_yscale("log"); axs[0].set_title("Floor area by outlet type")
sns.boxplot(data=o, x="LOC_GRP", y="HOURS", order=["City","Suburb","Town","Rural"], ax=axs[1], palette="pastel"); axs[1].set_title("Annual hours open by locale")
sns.scatterplot(data=o.sample(3000, random_state=3), x="SQ_FEET", y="HOURS", hue="LOC_GRP", alpha=.4, s=12, ax=axs[2]); axs[2].set_xscale("log"); axs[2].set_title("Floor area vs hours open")
plt.tight_layout(); plt.show()

### 3.5  Scoreboard (KPIs) + interactive Dashboard (drill-down by state)

In [ ]:
def scoreboard(d, title="United States – all public libraries"):
    k = [("Library systems", f"{len(d):,}"),
         ("Population served", f"{d.POPU_LSA.sum()/1e6:,.1f} M"),
         ("Visits", f"{d.VISITS.sum()/1e6:,.0f} M"),
         ("Circulation", f"{d.TOTCIR.sum()/1e6:,.0f} M"),
         ("Spend / resident", f"${d.TOTOPEXP.sum()/d.POPU_LSA.sum():,.0f}"),
         ("Visits / resident", f"{d.VISITS.sum()/d.POPU_LSA.sum():,.1f}"),
         ("Programs", f"{d.TOTPRO.sum()/1e6:,.2f} M"),
         ("Program attendance", f"{d.TOTATTEN.sum()/1e6:,.0f} M")]
    fig, axs = plt.subplots(1, len(k), figsize=(17, 1.9))
    for ax, (lab, val) in zip(axs, k):
        ax.axis("off"); ax.add_patch(plt.Rectangle((0,0),1,1, transform=ax.transAxes, color="#eef3f8", ec="#9fb3c8"))
        ax.text(.5, .58, val, ha="center", va="center", fontsize=16, weight="bold", color="#1f3b57")
        ax.text(.5, .22, lab, ha="center", va="center", fontsize=9, color="#555")
    fig.suptitle(title, fontsize=13, weight="bold", y=1.08); plt.show()

scoreboard(df)

In [ ]:
def dashboard(state="ALL"):
    d = df if state == "ALL" else df[df.STABR == state]
    scoreboard(d, f"Scoreboard – {state}")
    fig, axs = plt.subplots(2, 3, figsize=(18, 9))
    axs[0,0].hist(d.SPEND_PC.clip(upper=d.SPEND_PC.quantile(.98)), bins=40, color="steelblue")
    axs[0,0].set_title("Operating spend per resident (USD)")
    d.groupby("LOCALE_GRP")["VISITS_PC"].median().reindex(["City","Suburb","Town","Rural"]).plot.bar(ax=axs[0,1], color="teal", rot=0)
    axs[0,1].set_title("Median visits per resident by locale")
    d.groupby("SIZE_CLASS")["CIRC_PC"].median().plot.bar(ax=axs[0,2], color="slateblue", rot=25)
    axs[0,2].set_title("Median circulation per resident by size")
    sns.scatterplot(data=d, x="SPEND_PC", y="VISITS_PC", hue="LOCALE_GRP", alpha=.4, s=15, ax=axs[1,0])
    axs[1,0].set_xscale("log"); axs[1,0].set_yscale("log"); axs[1,0].set_title("Spend vs visits per resident")
    att = d[["K0_5ATTEN","K6_11ATTEN","YAATTEN","ADULTATTEN","GENATTEN"]].replace([-1,-3,-4,-9], np.nan).sum()
    att.index = ["Age 0-5","Age 6-11","Young adult","Adult","General"]
    att.plot.pie(ax=axs[1,1], autopct="%1.0f%%", startangle=90, ylabel=""); axs[1,1].set_title("Program attendance by audience")
    top = (d.groupby("STABR").VISITS.sum()/d.groupby("STABR").POPU_LSA.sum()).sort_values(ascending=False).head(10) if state=="ALL" \
          else d.nlargest(10, "VISITS").set_index("LIBNAME").VISITS_PC
    top.iloc[::-1].plot.barh(ax=axs[1,2], color="coral"); axs[1,2].set_title("Top-10 states: visits per resident" if state=="ALL" else "Top-10 libraries: visits per resident")
    plt.tight_layout(); plt.show()

dashboard("ALL")

In [ ]:
# Drill-down: change the state code and re-run, e.g. "NY", "CA", "TX", "MH" ...
dashboard("NY")

# Optional interactive version (works if ipywidgets is installed):
try:
    import ipywidgets as w
    w.interact(dashboard, state=w.Dropdown(options=["ALL"] + sorted(df.STABR.unique()), value="ALL"))
except ImportError:
    print("ipywidgets not installed – use dashboard('XX') for drill-down instead.")

# PHASE 4 — Data Interpretation & Storytelling

### 4.1  Evidence table – numbers that back the story

In [ ]:
tiny   = (df.POPU_LSA < 2500).mean()*100
under10 = (df.POPU_LSA < 10000).mean()*100
share_top = df.nlargest(int(len(df)*0.10), "POPU_LSA").POPU_LSA.sum() / df.POPU_LSA.sum() * 100
loc_tbl = df.groupby("LOCALE_GRP").agg(libraries=("LIBNAME","count"),
              median_pop=("POPU_LSA","median"), spend_pc=("SPEND_PC","median"),
              visits_pc=("VISITS_PC","median"), circ_pc=("CIRC_PC","median"),
              hours=("HRS_OPEN","median")).reindex(["City","Suburb","Town","Rural"])
df["STAFF_PER_10K"] = df.TOTSTAFF / df.POPU_LSA * 10_000
size_tbl = df.groupby("SIZE_CLASS").agg(libraries=("LIBNAME","count"), spend_pc=("SPEND_PC","median"),
              visits_pc=("VISITS_PC","median"), staff_per_10k=("STAFF_PER_10K","median"))
r_spend_visit = stats.spearmanr(df.SPEND_PC, df.VISITS_PC)[0]
r_hours_visit = stats.spearmanr(df.HRS_OPEN, df.VISITS)[0]

print(f"• {under10:.0f} % of library systems serve fewer than 10,000 people ({tiny:.0f} % fewer than 2,500)")
print(f"• The largest 10 % of systems serve {share_top:.0f} % of the population")
print(f"• Spearman ρ  spend/resident ↔ visits/resident = {r_spend_visit:.2f}")
print(f"• Spearman ρ  hours open ↔ visits             = {r_hours_visit:.2f}")
print(f"• Visits/resident: simple mean {df.VISITS_PC.mean():.2f} vs population-weighted {np.average(df.VISITS_PC, weights=df.POPU_LSA):.2f}")
display(loc_tbl.round(2)); display(size_tbl.round(2))

### 4.2  Insight extraction — *why* the patterns exist
*(The numbers below come from my run on the FY22 files — re-check them against your own output in 4.1 and edit if yours differ.)*

| What the graph shows | Why it happens | Value for the domain |
|---|---|---|
| Every size/usage variable is **extremely right-skewed** (moment skew 10–60, excess kurtosis in the hundreds; Q-Q plots bend away from the line; log10 fixes most of it). | Library systems range from one-room rural libraries to metropolitan systems; size grows multiplicatively, not additively. | Use **medians, log-scales and per-capita ratios**; a plain mean is dominated by a few giants. |
| **Weighted ≠ simple mean**: visits per resident ≈ 3.3 (simple) vs ≈ 2.0 (population-weighted). | 57 % of systems serve < 10,000 people, yet the largest 10 % of systems serve ≈ 71 % of residents. | "Typical library" and "typical resident" give different answers — always state which one you report. |
| **Small libraries do more per resident**: Tiny (< 2.5k) systems have the highest median spend (≈ $49) and visits (≈ 3.1) per resident and ≈ 8.8 staff per 10k people, vs ≈ 3.3 staff for Metro systems. | Fixed costs (a building, minimum opening hours, at least one staff member) are spread over very few residents; in small towns the library is often the only community hub. | Per-capita funding gaps are **not** only about rural vs urban; economies of scale mean large systems can serve people with fewer staff per head. |
| **Hours open ↔ visits** have a strong rank correlation (ρ ≈ 0.82); spend/resident ↔ visits/resident is moderate (ρ ≈ 0.65). | Annual hours is a *sum over all outlets*, so it rises with system size, and visits rise with size too — partly a size effect. | Opening hours matter, but compare per-outlet or per-capita before concluding "more hours cause more visits". |
| **10–15 % IQR outliers** in every measure. | They are real large systems, not data errors. | **Do not delete** them; use robust statistics or separate dashboards for metro systems. |
| **Regression imputation** (RMSE ≈ 0.87 M) beats interpolation (≈ 1.06 M) and median (≈ 5.45 M) for suppressed staff expenditure. | Staff expenditure is closely tied to total spend and staff FTE. | Better imputation for the confidential (-9) fields than a plain median. |

### 4.3  Contextualisation (domain application)
*Public-sector / social-infrastructure analytics:* the same pipeline can be applied to **healthcare** (clinics per population), **education** (schools vs enrolment) or **disaster management** (shelter capacity vs population density).
Libraries are also **emergency information hubs** (Wi-Fi, charging, cooling centres) – the Wi-Fi/PC-use variables show which regions already have this digital capacity.

### 4.4  Actionable recommendations
1. **Benchmark with per-capita and median metrics**, not raw totals – use the size classes (Tiny … Metro) to compare *like with like*.
2. **Target funding by size class and per-capita gaps**, not by locale label alone – use the dashboard drill-down to find states/libraries with the lowest visits per resident; test whether longer opening hours raise visits (strong association, ρ ≈ 0.8, but confounded by size).
3. **Invest in digital services (Wi-Fi, e-books, virtual programs)** where uptake per resident is low – they help small libraries reach people without more buildings.
4. **Fix data reporting**: suppressed (-9) values come from libraries with ≤ 2 FTE staff; use regression imputation and keep the imputation flags.
5. **Monitor outliers separately** – metro systems need their own dashboard because they distort national averages.

### 4.5  Conclusion
*(Write 3–4 lines in your own words: main finding, one surprise, one recommendation.)*